# Maze LMDP workflows

This notebook is the supported end-to-end walkthrough: arbitrary maze geometry, a flat first-exit solve, a fixed-subgoal hierarchy with online Z-iteration, the passive subgoal graph, and NMF-discovered core-gated soft subgoals with an interactive rollout.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm

from andrew_mlmdp import (
    LMDPEnvironment,
    Maze,
    ModelParameters,
    NMFDiscoveryParameters,
    SubgoalBasis,
    desirability_grid,
    discover_soft_subgoals,
    soft_hierarchy_parameters,
)
from andrew_mlmdp import (
    plotting as viz,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

## Maze and shared physical dynamics

In [ ]:
parameters = ModelParameters(
    upper_control_cost=0.85,
    lower_control_cost=0.1,
    off_target_reward=-1.0,
)
maze = Maze.from_file(PROJECT_ROOT / "mazes" / "four_rooms.txt")
environment = LMDPEnvironment(maze)
goal = (10, 9)

print(parameters)
print(f"maze shape: {maze.shape}")
print(f"free states: {len(maze.free_cells)}")
print(f"goal state: {maze.state_index(goal)}")

## Exact flat first-exit LMDP

The environment caches the physical passive dynamics. A flat solution contains the desirability, controlled policy, and rollout method for one goal.

In [ ]:
flat = environment.solve_flat(goal, parameters=parameters)
flat_grid = desirability_grid(maze, flat.desirability)
positive = flat_grid[np.isfinite(flat_grid) & (flat_grid > 0.0)]

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(
    flat_grid,
    cmap="viridis",
    norm=LogNorm(vmin=positive.min(), vmax=positive.max()),
)
ax.plot(goal[1], goal[0], marker="*", color="red", markersize=13)
ax.set(title="Exact flat desirability", xlabel="column", ylabel="row")
fig.colorbar(image, ax=ax, label="desirability (log scale)")
plt.show()

In [ ]:
viz.plot_controlled_dynamics(maze, flat.controlled, goal=goal)
plt.show()

flat_start = (3, 0)
flat_rollout = flat.rollout(flat_start, seed=0)
print(f"reached goal: {flat_rollout[-1] == goal}")
print(f"physical steps: {len(flat_rollout) - 1}")
viz.plot_trajectory(maze, flat_rollout, goal=goal)
plt.show()

## Fixed-subgoal two-layer hierarchy

Point subgoals are represented internally as one-hot profile columns. The same hierarchy implementation is used later for distributed profiles.

In [ ]:
subgoal_labels = ("A", "B", "C", "D", "E", "F")
subgoals = (
    (0, 0),
    (9, 2),
    (2, 3),
    (3, 7),
    (9, 7),
    (7, 9),
)
hierarchical_start = (3, 2)
point_basis = SubgoalBasis.from_locations(
    maze, subgoals, labels=subgoal_labels
)
hierarchy = environment.hierarchy(point_basis, parameters=parameters)
task = hierarchy.for_goal(goal)

### Task-independent passive subgoal graph

In [ ]:
subgoal_passive = hierarchy.passive_dynamics
fig, ax = plt.subplots(figsize=(7, 7))
viz.plot_subgoal_passive_dynamics(
    maze,
    subgoals,
    subgoal_passive,
    labels=subgoal_labels,
    ax=ax,
)
plt.show()
print(np.round(subgoal_passive, 4))

### Goal-conditioned matrices and task composition

In [ ]:
print("target order:", subgoal_labels + ("goal",))
print("lower passive:", task.lower_dynamics.passive.shape)
print("boundary basis Q_b:", task.task_basis.boundary_desirability.shape)
print("interior basis Z_i:", task.task_basis.interior_desirability.shape)
print("upper passive:", task.upper_dynamics.passive.shape)

abstract_labels = subgoal_labels + ("goal",)
matrix_maximum = max(
    task.upper_dynamics.passive.max(), task.upper_controlled.max()
)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, matrix, title in (
    (axes[0], task.upper_dynamics.passive, "Layer 2 passive"),
    (axes[1], task.upper_controlled, "Layer 2 controlled"),
):
    image = ax.imshow(matrix, cmap="viridis", vmin=0.0, vmax=matrix_maximum)
    ax.set_xticks(np.arange(len(subgoal_labels)), subgoal_labels)
    ax.set_yticks(np.arange(len(abstract_labels)), abstract_labels)
    ax.set(xlabel="current subgoal", ylabel="next abstract state", title=title)
fig.colorbar(image, ax=axes, label="transition probability")
plt.show()

In [ ]:
initial_plan = task.plan(hierarchical_start)
print(" target   passive  controlled  reward    weight")
for label, passive, controlled, reward, weight in zip(
    abstract_labels,
    initial_plan.passive_abstract,
    initial_plan.controlled_abstract,
    initial_plan.inpainted_rewards,
    initial_plan.weights,
):
    print(
        f" {label:>4}    {passive:7.3f}     {controlled:7.3f}"
        f"    {reward:7.3f}   {weight:7.3f}"
    )

composed_grid = desirability_grid(maze, initial_plan.physical_desirability)
positive = composed_grid[np.isfinite(composed_grid) & (composed_grid > 0.0)]
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(
    composed_grid,
    cmap="viridis",
    norm=LogNorm(vmin=positive.min(), vmax=positive.max()),
)
ax.plot(goal[1], goal[0], marker="*", color="red", markersize=13)
ax.set(title="Composed lower-layer desirability", xlabel="column", ylabel="row")
fig.colorbar(image, ax=ax, label="desirability (log scale)")
plt.show()

### Exact and online hierarchical rollouts

In [ ]:
hierarchical_rollout = task.rollout(hierarchical_start, seed=0)
print("status:", hierarchical_rollout.status)
print("physical steps:", hierarchical_rollout.physical_steps)
print("zero-time accesses:", hierarchical_rollout.accesses)

ax = viz.plot_trajectory(maze, hierarchical_rollout.trajectory, goal=goal)
for label, (row, column) in zip(subgoal_labels, subgoals):
    ax.scatter(column, row, s=60, facecolors="none", edgecolors="darkorange")
    ax.text(column + 0.13, row - 0.13, label, color="darkorange")
ax.set_title(
    f"Hierarchical rollout: {hierarchical_rollout.status} "
    f"({hierarchical_rollout.physical_steps} steps)"
)
plt.show()

In [ ]:
from IPython.display import HTML

rollout_animation = viz.animate_hierarchical_rollout(
    task,
    hierarchical_start,
    seed=0,
    max_steps=100,
    interval=450,
    subgoal_labels=subgoal_labels,
)
HTML(rollout_animation.to_jshtml())

In [ ]:
online_animation = viz.animate_hierarchical_rollout(
    task,
    hierarchical_start,
    goal_learning="online",
    z_sweeps_per_step=1,
    max_steps=100,
    seed=28,
    interval=450,
    subgoal_labels=subgoal_labels,
)
HTML(online_animation.to_jshtml())

In [ ]:
learned_goal = None
online_episodes = []
for episode_seed in range(5):
    episode = task.rollout(
        hierarchical_start,
        goal_learning="online",
        initial_goal_desirability=learned_goal,
        z_sweeps_per_step=1,
        max_steps=100,
        seed=episode_seed,
    )
    online_episodes.append(episode)
    learned_goal = episode.final_goal_desirability

for index, episode in enumerate(online_episodes, start=1):
    print(
        f"episode {index}: {episode.status}, "
        f"{episode.physical_steps} physical steps, "
        f"{episode.z_iterations} Z sweeps"
    )

### Interactive fixed-subgoal composition

Drag the agent and goal markers to inspect the fixed-basis task blend.

In [ ]:
%matplotlib widget
interactive_figure = viz.plot_interactive_subgoal_desirability(
    task,
    hierarchical_start,
    subgoal_labels=subgoal_labels,
)
interactive_figure

## NMF-discovered core-gated soft subgoals

Every requested rank is fitted once. The rank-eight result below is reused directly after plotting the rank diagnostics.

In [ ]:
%matplotlib inline
soft_study = discover_soft_subgoals(
    environment,
    ranks=tuple(range(2, 13)),
    parameters=NMFDiscoveryParameters(),
    seed=0,
)
viz.plot_soft_subtask_rank_diagnostics(soft_study.diagnostics)
soft_discovery = soft_study.result(8)
print(
    f"normalized KL error: {soft_discovery.reconstruction_error:.3f}; "
    f"{soft_discovery.n_iter} iterations; "
    f"converged={soft_discovery.converged}"
)
viz.plot_soft_subtasks(soft_discovery)

In [ ]:
soft_basis = SubgoalBasis.from_profiles(
    maze,
    soft_discovery.profiles,
    core_threshold=0.8,
    core_exponent=1.0,
)
soft_template = environment.hierarchy(
    soft_basis,
    parameters=soft_hierarchy_parameters(k=8, upper_control_cost=0.3),
    include_goal_component_while_active=False,
)

### Interactive soft rollout

Drag the green start circle and red goal star to stage new free cells, then press **Recompute rollout**. Recompute builds only the cached goal-conditioned task and samples one trajectory; it does not rerun NMF or reapply the core gate.

In [ ]:
%matplotlib widget
from IPython.display import display

soft_player = viz.plot_interactive_soft_hierarchical_rollout(
    soft_template,
    hierarchical_start,
    goal,
    max_steps=100,
    max_abstract_accesses=100,
    seed=0,
)
display(soft_player.controls)
plt.show()